# 04 — Claude Tools

Build and test the 5 agent tools individually before wiring them into LangGraph (notebook 05).

**Tools to build:**
1. `vector_search` — question + Pinecone retrieval + Claude answer with timestamps
2. `summarize_video` — all chunks to Claude, cache result in Pinecone
3. `get_topics` — extract topics with timestamps, cache in Pinecone
4. `compare_videos` — multi-namespace retrieval + Claude comparison
5. `get_metadata` — direct Pinecone fetch (no Claude needed)

**Measure:** token usage and cost per tool call, compare with COST_BREAKDOWN.md estimates.

In [1]:
import os
import json
import time
from dotenv import load_dotenv
from pinecone import Pinecone
from anthropic import Anthropic

load_dotenv()

# Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index(os.getenv("PINECONE_INDEX_NAME", "askthevideo"))

# Anthropic
client = Anthropic()
CLAUDE_MODEL = "claude-sonnet-4-6"

# Test video from notebook 03
VIDEO_ID = "e-gwvmhyU7A"

# Verify connections
stats = index.describe_index_stats()
print(f"Pinecone: {stats.namespaces.get(VIDEO_ID, {})}")
print(f"Claude model: {CLAUDE_MODEL}")

Pinecone: {'vector_count': 93}
Claude model: claude-sonnet-4-6


## 1. Tool: vector_search

Question → embed → Pinecone retrieval → build context with timestamped text → Claude answers with citations.

In [2]:
EMBED_MODEL = "llama-text-embed-v2"
SENTINEL_VECTOR = [1e-7] + [0.0] * 1023

def embed_texts(texts, input_type="passage"):
    all_embs = []
    for i in range(0, len(texts), 50):
        batch = texts[i:i+50]
        embs = pc.inference.embed(
            model=EMBED_MODEL, inputs=batch,
            parameters={"input_type": input_type, "truncate": "END"},
        )
        all_embs.extend([e.values for e in embs])
    return all_embs

def query_chunks(question, video_id, top_k=5):
    emb = embed_texts([question], input_type="query")[0]
    results = index.query(
        vector=emb, namespace=video_id,
        top_k=top_k, include_metadata=True,
    )
    return [
        {"score": m.score, "id": m.id, **m.metadata}
        for m in results.matches
        if m.metadata.get("type") != "metadata" # this is only used for local testing, before bug was discovered
        #if m.metadata.get("type") == "chunk" # this is updated field for production after bug discovered at later stage
    ]

print("Helpers loaded")

Helpers loaded


In [3]:
def vector_search(question: str, video_ids: list[str]) -> dict:
    """Search video transcripts and answer with Claude."""

    # Retrieve chunks from each selected video
    all_chunks = []
    per_video_k = max(3, 10 // len(video_ids))
    for vid in video_ids:
        chunks = query_chunks(question, vid, top_k=per_video_k)
        all_chunks.extend(chunks)

    if not all_chunks:
        return {"answer": "No relevant content found.", "usage": None}

    # Sort by score, take top 10
    all_chunks.sort(key=lambda c: c["score"], reverse=True)
    all_chunks = all_chunks[:10]

    # Build context from timestamped text
    context_parts = []
    for c in all_chunks:
        header = f"[{c['start_display']}–{c['end_display']}] ({c['video_url']})"
        context_parts.append(f"{header}\n{c['text_timestamped']}")
    context = "\n\n---\n\n".join(context_parts)

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system="You answer questions about YouTube videos using transcript excerpts. "
               "Always reference specific timestamps from the excerpts. "
               "If the excerpts don't contain relevant information, say so.",
        messages=[{
            "role": "user",
            "content": f"Transcript excerpts:\n\n{context}\n\n---\n\nQuestion: {question}",
        }],
    )

    return {
        "answer": response.content[0].text,
        "usage": {
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
        },
        "chunks_used": len(all_chunks),
    }

print("vector_search defined")

vector_search defined


In [4]:
%%time

result = vector_search("What do they think about Perplexity AI?", [VIDEO_ID])

print(f"Chunks used: {result['chunks_used']}")
print(f"Tokens: {result['usage']}")
print(f"\nAnswer:\n{result['answer']}")

Chunks used: 10
Tokens: {'input_tokens': 8726, 'output_tokens': 425}

Answer:
Based on the transcript excerpts, both Lex Fridman and Aravind Srinivas (CEO of Perplexity) hold quite positive views about Perplexity AI, though they acknowledge its limitations.

## Strengths They Highlight

**As an "Answer Engine":** At [1:53], Aravind describes Perplexity as best understood as an "answer engine" — combining traditional search with LLMs to produce cited, well-formatted answers, similar to how academics write papers with citations backing every sentence.

**Knowledge Discovery:** At [7:14], Aravind prefers calling it a "knowledge discovery engine," emphasizing that the journey doesn't end with an answer but continues through related questions at the bottom, expanding curiosity infinitely.

**Intent Understanding:** At [38:20–40:14], both agree Perplexity is good at understanding poorly constructed queries, noting the product should work even when users are lazy or imprecise — doing "the mag

### vector_search results

- **Quality:** Strong — specific timestamps, relevant citations, structured answer
- **Tokens:** ~8.7K input + ~450 output = ~9.2K total
- **Cost:** ~$0.033 per query ($3/MTok in + $15/MTok out)
- **Latency:** ~14s (mostly Claude generation)
- **Chunks:** 10 retrieved, all used in context

## 2. Tool: summarize_video

Fetch all chunks → send to Claude → cache result in Pinecone as sentinel-vector record.

In [5]:
def summarize_video(video_id: str) -> dict:
    """Generate or retrieve cached video summary."""

    # Check cache first
    cached = index.fetch(ids=[f"{video_id}_summary"], namespace=video_id)
    vec = cached.vectors.get(f"{video_id}_summary")
    if vec and vec.metadata.get("text"):
        return {
            "summary": vec.metadata["text"],
            "cached": True,
            "usage": None,
        }

    # Fetch all chunks (sorted by chunk_index)
    # Use a dummy query — we want ALL chunks, not semantic match
    # Fetch by listing namespace vectors
    stats = index.describe_index_stats()
    total = int(stats.namespaces.get(video_id, {}).get("vector_count", 0))

    # Fetch all chunk IDs
    chunk_ids = [f"{video_id}_chunk_{i:03d}" for i in range(total)]
    fetched = index.fetch(ids=chunk_ids, namespace=video_id)

    # Sort by chunk_index and build full transcript
    chunks = []
    for cid in chunk_ids:
        vec = fetched.vectors.get(cid)
        if vec and vec.metadata.get("type", "chunk") == "chunk":
            chunks.append(vec.metadata)

    chunks.sort(key=lambda c: c.get("chunk_index", 0))
    full_text = "\n\n".join(
        f"[{c['start_display']}–{c['end_display']}]\n{c['text']}"
        for c in chunks
    )

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=2048,
        system="You summarise YouTube videos from their transcript. "
               "Provide: a one-paragraph overview, then 5-7 key points with timestamps. "
               "Be concise and specific.",
        messages=[{
            "role": "user",
            "content": f"Summarise this video transcript:\n\n{full_text}",
        }],
    )

    summary_text = response.content[0].text

    # Cache in Pinecone
    index.upsert(
        vectors=[{
            "id": f"{video_id}_summary",
            "values": SENTINEL_VECTOR,
            "metadata": {
                "type": "summary",
                "video_id": video_id,
                "text": summary_text,
            },
        }],
        namespace=video_id,
    )

    return {
        "summary": summary_text,
        "cached": False,
        "usage": {
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
        },
        "chunks_sent": len(chunks),
    }

print("summarize_video defined")

summarize_video defined


In [6]:
%%time

result = summarize_video(VIDEO_ID)

print(f"Cached: {result['cached']}")
print(f"Chunks sent: {result.get('chunks_sent', 'n/a')}")
print(f"Tokens: {result.get('usage', 'n/a')}")
print(f"\nSummary:\n{result['summary']}")

Cached: True
Chunks sent: n/a
Tokens: None

Summary:
## Lex Fridman Podcast: Aravind Srinivas (CEO of Perplexity)

Aravind Srinivas, CEO of Perplexity AI, joins Lex Fridman to discuss how Perplexity combines search and LLMs into an "answer engine," the business dynamics of challenging Google, the history of LLMs, and the future of AI reasoning and knowledge discovery. The conversation blends deep technical detail with startup philosophy and big-picture speculation about AGI.

---

**Key Points:**

1. [1:53](https://youtu.be/e-gwvmhyU7A?t=113) **How Perplexity works** — Perplexity is an "answer engine": it retrieves relevant web sources, extracts paragraphs, and instructs an LLM to write a cited, academic-style answer. The founding insight came from academic writing, where every sentence must have a citation to prevent hallucination.

2. [16:01](https://youtu.be/e-gwvmhyU7A?t=961) **Why Perplexity doesn't try to beat Google at its own game** — Rather than rebuilding Google's 10-blue-lin

In [7]:
%%time

result2 = summarize_video(VIDEO_ID)

print(f"Cached: {result2['cached']}")
print(f"Tokens: {result2.get('usage', 'n/a')}")
print(f"\nFirst 200 chars:\n{result2['summary'][:200]}")

Cached: True
Tokens: None

First 200 chars:
## Lex Fridman Podcast: Aravind Srinivas (CEO of Perplexity)

Aravind Srinivas, CEO of Perplexity AI, joins Lex Fridman to discuss how Perplexity combines search and LLMs into an "answer engine," the 
CPU times: user 9 ms, sys: 2.74 ms, total: 11.7 ms
Wall time: 170 ms


### summarize_video results

| Metric | Fresh | Cached |
|---|---|---|
| Latency | ~22s | ~0.6s |
| Input tokens | 43,518 | 0 |
| Output tokens | 724 | 0 |
| Cost | ~$0.14 | $0.00 |

Caching pays for itself after 1 use. Most videos will be summarised once and served from cache.

## 3. Tool: get_topics

Same pattern as summarize — fetch all chunks, Claude extracts topics with timestamps, cache result.

In [8]:
def get_topics(video_id: str) -> dict:
    """Generate or retrieve cached topic list."""

    # Check cache
    cached = index.fetch(ids=[f"{video_id}_topics"], namespace=video_id)
    vec = cached.vectors.get(f"{video_id}_topics")
    if vec and vec.metadata.get("text"):
        return {
            "topics": vec.metadata["text"],
            "cached": True,
            "usage": None,
        }

    # Fetch all chunks
    stats = index.describe_index_stats()
    total = int(stats.namespaces.get(video_id, {}).get("vector_count", 0))
    chunk_ids = [f"{video_id}_chunk_{i:03d}" for i in range(total)]
    fetched = index.fetch(ids=chunk_ids, namespace=video_id)

    chunks = []
    for cid in chunk_ids:
        vec = fetched.vectors.get(cid)
        if vec and vec.metadata.get("type", "chunk") == "chunk":
            chunks.append(vec.metadata)

    chunks.sort(key=lambda c: c.get("chunk_index", 0))
    full_text = "\n\n".join(
        f"[{c['start_display']}–{c['end_display']}]\n{c['text']}"
        for c in chunks
    )

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system="You extract the main topics from a YouTube video transcript. "
               "Return a numbered list of 8-12 topics, each with a timestamp range "
               "and a one-sentence description. Format: '1. [MM:SS-MM:SS] Topic — description'",
        messages=[{
            "role": "user",
            "content": f"Extract the main topics from this transcript:\n\n{full_text}",
        }],
    )

    topics_text = response.content[0].text

    # Cache
    index.upsert(
        vectors=[{
            "id": f"{video_id}_topics",
            "values": SENTINEL_VECTOR,
            "metadata": {
                "type": "topics",
                "video_id": video_id,
                "text": topics_text,
            },
        }],
        namespace=video_id,
    )

    return {
        "topics": topics_text,
        "cached": False,
        "usage": {
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
        },
    }

print("get_topics defined")

get_topics defined


In [9]:
%%time

result = get_topics(VIDEO_ID)

print(f"Cached: {result['cached']}")
print(f"Tokens: {result.get('usage', 'n/a')}")
print(f"\nTopics:\n{result['topics']}")

Cached: True
Tokens: None

Topics:
Here are the main topics from this transcript:

1. [0:00](https://youtu.be/e-gwvmhyU7A?t=0) Introduction to Perplexity — Lex introduces Aravind Srinivas and Perplexity as an answer engine that combines search and LLMs with source citations.

2. [1:53](https://youtu.be/e-gwvmhyU7A?t=113) How Perplexity Works — Aravind explains the citation-first philosophy inspired by academic writing and how it forces factual grounding.

3. [7:59](https://youtu.be/e-gwvmhyU7A?t=479) Knowledge Discovery vs. Search — Perplexity is framed as a knowledge discovery engine driven by human curiosity, with related questions guiding users deeper.

4. [16:01](https://youtu.be/e-gwvmhyU7A?t=961) Competing with Google — Aravind argues disruption comes from rethinking the UI entirely rather than playing Google's 10-blue-links game.

5. [18:03](https://youtu.be/e-gwvmhyU7A?t=1083) Google's AdWords Business Model — The conversation breaks down how Google's auction-based ad system wo

In [10]:
%%time
# Quick cache check
result2 = get_topics(VIDEO_ID)
print(f"Cached: {result2['cached']}, latency measured by %%time")

Cached: True, latency measured by %%time
CPU times: user 3.93 ms, sys: 1.87 ms, total: 5.8 ms
Wall time: 130 ms


## 4. Tool: compare_videos

We only have one video loaded, so we'll test the mechanics with a single video and validate the multi-namespace pattern works. Real multi-video testing happens in notebook 05 with the agent.

## 5. Tool: get_metadata

Direct Pinecone fetch — no Claude needed.

In [11]:
def compare_videos(question: str, video_ids: list[str]) -> dict:
    """Compare what multiple videos say about a topic."""

    all_chunks = []
    per_video_k = max(3, 10 // len(video_ids))

    for vid in video_ids:
        chunks = query_chunks(question, vid, top_k=per_video_k)
        # Tag each chunk with its video_id for attribution
        for c in chunks:
            c["video_id"] = vid
        all_chunks.extend(chunks)

    if not all_chunks:
        return {"answer": "No relevant content found.", "usage": None}

    all_chunks.sort(key=lambda c: c["score"], reverse=True)

    # Build context grouped by video
    by_video = {}
    for c in all_chunks:
        vid = c["video_id"]
        if vid not in by_video:
            by_video[vid] = []
        by_video[vid].append(c)

    context_parts = []
    for vid, chunks in by_video.items():
        # Get video title from first chunk or fallback
        header = f"=== Video: {vid} ==="
        excerpts = []
        for c in chunks:
            excerpts.append(
                f"[{c['start_display']}–{c['end_display']}] ({c['video_url']})\n"
                f"{c['text_timestamped']}"
            )
        context_parts.append(header + "\n\n" + "\n\n".join(excerpts))

    context = "\n\n" + "=" * 40 + "\n\n".join(context_parts)

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1536,
        system="You compare what different YouTube videos say about a topic. "
               "Highlight similarities and differences. Reference specific timestamps. "
               "If only one video is provided, summarise what it says about the topic.",
        messages=[{
            "role": "user",
            "content": f"Compare these videos on the topic:\n\n"
                       f"Question: {question}\n\n{context}",
        }],
    )

    return {
        "answer": response.content[0].text,
        "usage": {
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
        },
        "videos_queried": len(video_ids),
        "chunks_used": len(all_chunks),
    }


def get_metadata(video_id: str) -> dict:
    """Fetch video metadata from Pinecone. No Claude call needed."""
    result = index.fetch(ids=[f"{video_id}_metadata"], namespace=video_id)
    vec = result.vectors.get(f"{video_id}_metadata")
    if vec:
        return {"metadata": dict(vec.metadata), "found": True}
    return {"metadata": None, "found": False}


print("compare_videos and get_metadata defined")

compare_videos and get_metadata defined


In [12]:
%%time

# Test compare_videos (single video — validates mechanics)
print("=== compare_videos ===")
result = compare_videos("How do they describe the history of transformers?", [VIDEO_ID])
print(f"Videos: {result['videos_queried']}, Chunks: {result['chunks_used']}")
print(f"Tokens: {result['usage']}")
print(f"\nAnswer (first 500 chars):\n{result['answer'][:500]}")

=== compare_videos ===
Videos: 1, Chunks: 10
Tokens: {'input_tokens': 8882, 'output_tokens': 1131}

Answer (first 500 chars):
## History of Transformers – Single Video Summary (e-gwvmhyU7A)

Since only one video is provided, here is a summary of how Aravind Srinivas describes the history of transformers across the relevant segments.

---

### The Pre-Transformer Era: RNNs and Attention

At **[1:04:43]**, Aravind traces the origins back to sequence-to-sequence learning with RNNs. He credits **Ilya Sutskever** with the first paper showing a simple scaled-up RNN could beat phrase-based machine translation systems — but no
CPU times: user 22 ms, sys: 24.2 ms, total: 46.2 ms
Wall time: 27.7 s


In [13]:
# Test get_metadata
print("=== get_metadata ===")
result = get_metadata(VIDEO_ID)
print(f"Found: {result['found']}")
for k, v in result['metadata'].items():
    print(f"  {k}: {v}")

# Test non-existent
result2 = get_metadata("fake_video_id")
print(f"\nFake video found: {result2['found']}")

=== get_metadata ===
Found: True
  channel: Test Channel
  chunk_count: 90.0
  duration_display: 3:02:06
  duration_seconds: 10926.963
  ingested_at: 2026-03-10T13:24:21Z
  type: metadata
  video_id: e-gwvmhyU7A
  video_title: AI tools podcast (test video)

Fake video found: False


## 6. Cost comparison vs estimates

In [15]:
# Pricing: Sonnet 4.6 = $3/MTok input, $15/MTok output
IN_COST = 3.0 / 1_000_000
OUT_COST = 15.0 / 1_000_000

tools_measured = {
    "vector_search": {"in": 8726, "out": 454},
    "summarize_video (fresh)": {"in": 43518, "out": 724},
    "summarize_video (cached)": {"in": 0, "out": 0},
    "get_topics (fresh)": {"in": 43537, "out": 853},
    "get_topics (cached)": {"in": 0, "out": 0},
    "compare_videos (1 video)": {"in": 8882, "out": 940},
    "get_metadata": {"in": 0, "out": 0},
}

print(f"{'Tool':<30} {'In tokens':>10} {'Out tokens':>10} {'Cost':>10}")
print("-" * 65)
total_cost = 0
for tool, usage in tools_measured.items():
    cost = usage["in"] * IN_COST + usage["out"] * OUT_COST
    total_cost += cost
    print(f"{tool:<30} {usage['in']:>10,} {usage['out']:>10,} ${cost:>8.4f}")

print("-" * 65)
print(f"\n10 questions session estimate (realistic mix):")
print(f"  7x vector_search:  ${7 * (8726*IN_COST + 454*OUT_COST):.3f}")
print(f"  1x summarize:      ${43518*IN_COST + 724*OUT_COST:.3f}")
print(f"  1x get_topics:     ${43537*IN_COST + 853*OUT_COST:.3f}")
print(f"  1x compare/meta:   ${8882*IN_COST + 940*OUT_COST:.3f}")
session_total = (
    7 * (8726*IN_COST + 454*OUT_COST) +
    1 * (43518*IN_COST + 724*OUT_COST) +
    1 * (43537*IN_COST + 853*OUT_COST) +
    1 * (8882*IN_COST + 940*OUT_COST)
)
print(f"  TOTAL:             ${session_total:.3f}")

Tool                            In tokens Out tokens       Cost
-----------------------------------------------------------------
vector_search                       8,726        454 $  0.0330
summarize_video (fresh)            43,518        724 $  0.1414
summarize_video (cached)                0          0 $  0.0000
get_topics (fresh)                 43,537        853 $  0.1434
get_topics (cached)                     0          0 $  0.0000
compare_videos (1 video)            8,882        940 $  0.0407
get_metadata                            0          0 $  0.0000
-----------------------------------------------------------------

10 questions session estimate (realistic mix):
  7x vector_search:  $0.231
  1x summarize:      $0.141
  1x get_topics:     $0.143
  1x compare/meta:   $0.041
  TOTAL:             $0.556


## 7. Cost comparison: estimated vs actual

| Tool | Estimated (60-min video) | Actual (182-min video) | Notes |
|---|---|---|---|
| vector_search | $0.01-0.02 | $0.033 | 10 chunks with timestamped text = more context than estimated |
| summarize_video (fresh) | $0.05 | $0.141 | 3x longer video = 3x more input tokens. Scales linearly. |
| get_topics (fresh) | $0.05 | $0.143 | Same as summarize — full transcript sent |
| compare_videos | $0.02 | $0.041 | Single video test; 2-video would be ~2x |
| 10-question session | ~$0.15 | $0.56 | Actual video is 3x the 60-min reference |

**Why actuals are higher:**
1. Test video is 182 min, not 60 min — summarize/topics scale linearly with duration
2. vector_search sends 10 chunks (not 5) with full timestamped text — richer context but more tokens
3. Estimates assumed ~2-3K input for vector_search; actual is ~8.7K

**Adjusted estimate for a 60-min video:**
- summarize/topics: ~$0.05 (matches original estimate — 90 chunks / 3 = 30 chunks)
- vector_search: ~$0.03 (higher than estimated due to timestamped context)
- 10-question session: ~$0.25-0.30

Still well within budget. The 60-min duration cap for free users keeps costs controlled.

## 8. Production functions

Shared helper for summarize/topics (fetch all chunks + cache pattern), then all 5 tools.

In [16]:
# fix to path to be able to run production cell in notebook
import sys
sys.path.insert(0, "..")

In [17]:
# @export src/tools.py
import os
import time
import logging
from anthropic import Anthropic, APIError
from pinecone import Pinecone

from src.metrics import record_tokens
from src.errors import send_discord_alert

logger = logging.getLogger(__name__)

CLAUDE_MODEL = "claude-sonnet-4-6"
EMBED_MODEL = "llama-text-embed-v2"
EMBED_BATCH_SIZE = 50
SENTINEL_VECTOR = [1e-7] + [0.0] * 1023


def _get_clients():
    """Initialise and return Anthropic + Pinecone clients."""
    anthropic_client = Anthropic()
    pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
    index = pc.Index(os.getenv("PINECONE_INDEX_NAME", "askthevideo"))
    return anthropic_client, pc, index


def _embed_texts(pc, texts, input_type="passage"):
    """Embed texts via Pinecone Inference API."""
    all_embs = []
    for i in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = texts[i:i + EMBED_BATCH_SIZE]
        embs = pc.inference.embed(
            model=EMBED_MODEL, inputs=batch,
            parameters={"input_type": input_type, "truncate": "END"},
        )
        all_embs.extend([e.values for e in embs])
    return all_embs


def _query_chunks(pc, index, question, video_id, top_k=5):
    """Embed question and retrieve matching chunks from a namespace."""
    emb = _embed_texts(pc, [question], input_type="query")[0]
    results = index.query(
        vector=emb, namespace=video_id,
        top_k=top_k, include_metadata=True,
    )
    return [
        {"score": m.score, "id": m.id, **m.metadata}
        for m in results.matches
        if m.metadata.get("type") == "chunk"
    ]


def _fetch_all_chunks(index, video_id):
    """Fetch all chunk records for a video, sorted by chunk_index."""
    stats = index.describe_index_stats()
    total = int(stats.namespaces.get(video_id, {}).get("vector_count", 0))
    chunk_ids = [f"{video_id}_chunk_{i:03d}" for i in range(total)]
    fetched = index.fetch(ids=chunk_ids, namespace=video_id)
    chunks = []
    for cid in chunk_ids:
        vec = fetched.vectors.get(cid)
        if vec and vec.metadata.get("type", "chunk") == "chunk":
            chunks.append(dict(vec.metadata))
    chunks.sort(key=lambda c: c.get("chunk_index", 0))
    return chunks


def _build_full_text(chunks):
    """Build plain-text transcript from chunks (with timestamp headers and video URLs)."""
    return "\n\n".join(
        f"[{c['start_display']}–{c['end_display']}]({c['video_url']})\n{c['text']}"
        for c in chunks
    )


def _claude_create(client, **kwargs):
    """Call client.messages.create with Anthropic error alerting."""
    try:
        response = client.messages.create(**kwargs)
        record_tokens(response.usage.input_tokens, response.usage.output_tokens)
        return response
    except APIError as e:
        send_discord_alert(
            f"Anthropic API error: {e.status_code} {type(e).__name__}: {str(e)[:200]}",
            alert_type="anthropic_api",
        )
        raise


def _fetch_or_generate_cached(index, client, video_id, record_suffix, system_prompt, user_prompt_prefix):
    """Shared pattern: check Pinecone cache, generate with Claude if missing, cache result."""
    record_id = f"{video_id}_{record_suffix}"

    # Check cache
    cached = index.fetch(ids=[record_id], namespace=video_id)
    vec = cached.vectors.get(record_id)
    if vec and vec.metadata.get("text"):
        return {"text": vec.metadata["text"], "cached": True, "usage": None}

    # Fetch all chunks and build transcript
    chunks = _fetch_all_chunks(index, video_id)
    if not chunks:
        return {"text": "No transcript data found.", "cached": False, "usage": None}

    full_text = _build_full_text(chunks)

    response = _claude_create(
        client,
        model=CLAUDE_MODEL,
        max_tokens=2048,
        system=system_prompt,
        messages=[{"role": "user", "content": f"{user_prompt_prefix}\n\n{full_text}"}],
    )

    result_text = response.content[0].text

    # Cache
    index.upsert(
        vectors=[{
            "id": record_id,
            "values": SENTINEL_VECTOR,
            "metadata": {
                "type": record_suffix,
                "video_id": video_id,
                "text": result_text,
            },
        }],
        namespace=video_id,
    )

    return {
        "text": result_text,
        "cached": False,
        "usage": {
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
        },
    }


# ── Tool 1: vector_search ──

def vector_search(pc, index, client, question, video_ids):
    """Search transcript chunks and answer with Claude."""
    all_chunks = []
    per_video_k = max(3, 10 // len(video_ids))
    for vid in video_ids:
        all_chunks.extend(_query_chunks(pc, index, question, vid, top_k=per_video_k))

    if not all_chunks:
        return {"answer": "No relevant content found.", "usage": None}

    all_chunks.sort(key=lambda c: c["score"], reverse=True)
    all_chunks = all_chunks[:10]

    context_parts = []
    for c in all_chunks:
        header = f"[{c['start_display']}–{c['end_display']}] ({c['video_url']})"
        context_parts.append(f"{header}\n{c['text_timestamped']}")
    context = "\n\n---\n\n".join(context_parts)

    response = _claude_create(
        client,
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system="You answer questions about YouTube videos using transcript excerpts. "
               "Always reference specific timestamps as clickable markdown links, e.g. [2:30](https://youtu.be/ID?t=150). "
               "Use the video URLs provided in the excerpt headers. "
               "If the excerpts don't contain relevant information, say so.",
        messages=[{
            "role": "user",
            "content": f"Transcript excerpts:\n\n{context}\n\n---\n\nQuestion: {question}",
        }],
    )

    return {
        "answer": response.content[0].text,
        "usage": {
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
        },
        "chunks_used": len(all_chunks),
    }


# ── Tool 2: summarize_video ──

def summarize_video(index, client, video_id):
    """Generate or retrieve cached video summary."""
    result = _fetch_or_generate_cached(
        index, client, video_id,
        record_suffix="summary",
        system_prompt=(
            "You summarise YouTube videos from their transcript. "
            "Provide: a one-paragraph overview, then 5-7 key points with timestamps. "
            "Format timestamps as clickable markdown links using the URLs from the transcript headers, "
            "e.g. [2:30](https://youtu.be/ID?t=150). Be concise and specific."
        ),
        user_prompt_prefix="Summarise this video transcript:",
    )
    return {"summary": result["text"], "cached": result["cached"], "usage": result["usage"]}


# ── Tool 3: get_topics ──

def get_topics(index, client, video_id):
    """Generate or retrieve cached topic list."""
    result = _fetch_or_generate_cached(
        index, client, video_id,
        record_suffix="topics",
        system_prompt=(
            "You extract the main topics from a YouTube video transcript. "
            "Return a numbered list of 8-12 topics, each with a clickable timestamp link "
            "and a one-sentence description. Use the video URLs from the transcript headers. "
            "Format: '1. [MM:SS](https://youtu.be/ID?t=SECONDS) Topic — description'"
        ),
        user_prompt_prefix="Extract the main topics from this transcript:",
    )
    return {"topics": result["text"], "cached": result["cached"], "usage": result["usage"]}


# ── Tool 4: compare_videos ──

def compare_videos(pc, index, client, question, video_ids):
    """Compare what multiple videos say about a topic."""
    all_chunks = []
    per_video_k = max(3, 10 // len(video_ids))

    for vid in video_ids:
        chunks = _query_chunks(pc, index, question, vid, top_k=per_video_k)
        for c in chunks:
            c["video_id"] = vid
        all_chunks.extend(chunks)

    if not all_chunks:
        return {"answer": "No relevant content found.", "usage": None}

    all_chunks.sort(key=lambda c: c["score"], reverse=True)

    by_video = {}
    for c in all_chunks:
        vid = c["video_id"]
        if vid not in by_video:
            by_video[vid] = []
        by_video[vid].append(c)

    context_parts = []
    for vid, chunks in by_video.items():
        header = f"=== Video: {vid} ==="
        excerpts = [
            f"[{c['start_display']}–{c['end_display']}] ({c['video_url']})\n{c['text_timestamped']}"
            for c in chunks
        ]
        context_parts.append(header + "\n\n" + "\n\n".join(excerpts))

    context = ("\n\n" + "=" * 40 + "\n\n").join(context_parts)

    response = _claude_create(
        client,
        model=CLAUDE_MODEL,
        max_tokens=1536,
        system="You compare what different YouTube videos say about a topic. "
               "Highlight similarities and differences. "
               "Reference specific timestamps as clickable markdown links using the video URLs from the excerpts, "
               "e.g. [2:30](https://youtu.be/ID?t=150). "
               "If only one video is provided, summarise what it says about the topic.",
        messages=[{
            "role": "user",
            "content": f"Compare these videos on the topic:\n\nQuestion: {question}\n\n{context}",
        }],
    )

    return {
        "answer": response.content[0].text,
        "usage": {
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
        },
        "videos_queried": len(video_ids),
        "chunks_used": len(all_chunks),
    }


# ── Tool 5: get_metadata ──

def get_metadata(index, video_id):
    """Fetch video metadata from Pinecone."""
    result = index.fetch(ids=[f"{video_id}_metadata"], namespace=video_id)
    vec = result.vectors.get(f"{video_id}_metadata")
    if vec:
        return {"metadata": dict(vec.metadata), "found": True}
    return {"metadata": None, "found": False}

In [24]:
# Re-ingest test video with "type": "chunk" metadata
from src.vectorstore import upsert_chunks, upsert_metadata_record
from src.chunking import chunk_transcript, format_time

# Delete old namespace and re-upsert
index.delete(delete_all=True, namespace=VIDEO_ID)

# Re-generate chunks (using chunking function from earlier in this notebook or import)
from src.chunking import chunk_transcript
from src.transcript import fetch_transcript

transcript = fetch_transcript(VIDEO_ID)
chunks = chunk_transcript(transcript["snippets"], VIDEO_ID)

# Upsert with proper "type": "chunk" metadata
count = upsert_chunks(pc, index, chunks, VIDEO_ID)
print(f"Upserted {count} chunks")

# Re-create metadata record
upsert_metadata_record(index, VIDEO_ID, {
    "video_title": "AI tools podcast (test video)",
    "channel": "Test Channel",
    "duration_seconds": transcript["duration_seconds"],
    "duration_display": format_time(transcript["duration_seconds"]),
    "chunk_count": len(chunks),
})
print(f"Metadata record created")
print(f"Done — {VIDEO_ID} re-ingested with type='chunk' on all records")

Upserted 90 chunks
Metadata record created
Done — e-gwvmhyU7A re-ingested with type='chunk' on all records


In [18]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [19]:
# Quick smoke test using production function signatures
client_a, pc_prod, index_prod = _get_clients()

# vector_search
r = vector_search(pc_prod, index_prod, client_a, "What is Perplexity?", [VIDEO_ID])
print(f"vector_search: {len(r['answer'])} chars, {r['usage']['input_tokens']} in tokens")

# summarize (should be cached from earlier)
r = summarize_video(index_prod, client_a, VIDEO_ID)
print(f"summarize_video: cached={r['cached']}")

# get_topics (should be cached)
r = get_topics(index_prod, client_a, VIDEO_ID)
print(f"get_topics: cached={r['cached']}")

# compare_videos
r = compare_videos(pc_prod, index_prod, client_a, "transformers history", [VIDEO_ID])
print(f"compare_videos: {r['chunks_used']} chunks, {r['usage']['input_tokens']} in tokens")

# get_metadata
r = get_metadata(index_prod, VIDEO_ID)
print(f"get_metadata: found={r['found']}, title={r['metadata']['video_title']}")

print("\n✅ All 5 production tools working")

vector_search: 1454 chars, 8676 in tokens
summarize_video: cached=True
get_topics: cached=True
compare_videos: 10 chunks, 8999 in tokens
get_metadata: found=True, title=AI tools podcast (test video)

✅ All 5 production tools working


## Summary

**What we built:**
- `_embed_texts()` — batch embed via Pinecone Inference
- `_query_chunks()` — embed + retrieve matching chunks
- `_fetch_all_chunks()` — fetch all chunk records for a video
- `_fetch_or_generate_cached()` — shared cache pattern for summarize/topics
- `vector_search()` — question answering with timestamp citations
- `summarize_video()` — full video summary with Pinecone caching
- `get_topics()` — topic extraction with timestamps and caching
- `compare_videos()` — multi-namespace retrieval + comparison
- `get_metadata()` — direct Pinecone fetch, no Claude call

**Production cells tagged:** 1 cell → `src/tools.py`

**Cost per tool call (182-min test video):**

| Tool | Input tokens | Output tokens | Cost | Latency |
|---|---|---|---|---|
| vector_search | ~8,700 | ~450 | $0.033 | ~14s |
| summarize_video (fresh) | ~43,500 | ~720 | $0.141 | ~22s |
| summarize_video (cached) | 0 | 0 | $0.000 | ~0.6s |
| get_topics (fresh) | ~43,500 | ~850 | $0.143 | ~23s |
| get_topics (cached) | 0 | 0 | $0.000 | ~0.6s |
| compare_videos (1 video) | ~8,900 | ~940 | $0.041 | ~26s |
| get_metadata | 0 | 0 | $0.000 | ~0.2s |

**10-question session estimate:** ~$0.56 (182-min video), ~$0.25-0.30 (60-min video)

**Key design patterns:**
- Shared `_fetch_or_generate_cached()` eliminates duplication between summarize and topics
- Sentinel vector `[1e-7, 0, ...]` for cache records (Pinecone rejects pure zeros)
- `input_type="query"` for questions, `"passage"` for documents
- Post-query filter removes metadata records from search results
- Timestamped text sent to Claude for precise timestamp citations

**Next:** Notebook 05 — Agent Routing (LangGraph agent with all 5 tools, routing accuracy, multi-turn conversation)